In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import spacy
import re
import nltk
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import pytextrank
from gensim.corpora import Dictionary
from gensim.models.ldamodel import LdaModel

ModuleNotFoundError: No module named 'gensim'

In [ ]:
!python -m pip install --upgrade numpy scipy pandas matplotlib seaborn --user


In [ ]:
df=pd.read_csv(r'C:\Users\user\Desktop\tasks\Predicting-Price-Moves-with-News-Sentiment\data\raw\raw_analyst_ratings.csv')
print(df)

In [ ]:
text_data = df['headline']
# Display descriptive statistics for the 'headline' column
print("Descriptive statistics for 'headline' column:")
print(df['headline'].describe())

# Check for missing values in the 'headline' column
print("\nMissing values in 'headline' column:")
print(df['headline'].isnull().sum())

# Count the number of articles per publisher
print(df.groupby("publisher")["headline"].value_counts())


In [ ]:
# Convert date column to datetime
df['date'] = pd.to_datetime(df['date'], format="%Y-%m-%d %H:%M:%S", errors='coerce')
daily_counts = df.groupby('date').size()  # number of articles per day
print(daily_counts)
# Daily trend
daily_counts.plot(figsize=(12,5), title="Articles Published per Day")
plt.xlabel("Date")
plt.ylabel("Number of Articles")
plt.show()


## Text Analysis(Topic Modeling)

In [ ]:
# # Download necessary NLTK data
nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")

# load spcy's english model
nlp=spacy.load("en_core_web_sm")

In [ ]:
# data processing
def text_processing(text):
  if isinstance(text,str):
    text=text.lower()
    text=text.translate(str.maketrans("","",string.punctuation))
    tokens=word_tokenize(text)
    stop_words=set(stopwords.words("english"))
    tokens=[word for word in tokens if word not in stop_words and len(word)>2]
    return tokens
  else:
    return []
# # Apply preprocessing to the 'headline' column
df["clean_headline"]=df["headline"].apply(text_processing)

In [ ]:
#  Create a dictionary from the processed headlines
dictionary=Dictionary(df["clean_headline"])

#  Create a corpus (a list of bag-of-words representations)
corpus=[dictionary.doc2bow(text) for text in df["clean_headline"]]

In [ ]:
df

In [11]:
# Train the LDA model
# You can adjust the number of topics (num_topics) as needed
lda_model=LdaModel(corpus,num_topics=5, id2word=dictionary, passes=15)

NameError: name 'LdaModel' is not defined

In [ ]:
# Print the topics found by the LDA model
print("\nTopics found by LDA model:")
for idx, topic in lda_model.print_topics(-1):
    print(f'Topic {idx}: {topic}')


In [ ]:
# Assign dominant topic to each headline
def get_dominant_topic(lda_model, bow):
    topics = lda_model.get_document_topics(bow)
    topics.sort(key=lambda x: x[1], reverse=True)
    if topics:
        return topics[0][0]
    return None

df['dominant_topic'] = [get_dominant_topic(lda_model, doc) for doc in corpus]

In [ ]:
# add PyTextRank to the spaCy pipeline
import spacy
import pytextrank

nlp = spacy.load("en_core_web_sm")

# Add PyTextRank to the pipeline
nlp.add_pipe("textrank")

# 3️⃣ Extract key phrases
phrases_list = []
for text in df['headline'].astype(str):
    doc = nlp(text)
    phrases = [phrase.text for phrase in doc._.phrases[:5]]  # top 5 phrases
    phrases_list.append(phrases)

df['key_phrases'] = phrases_list
df.head()

# Time Series Analysis

In [2]:
# Identify spikes
spikes = daily_counts[daily_counts > daily_counts.mean() + 2*daily_counts.std()]
print(spikes)


NameError: name 'daily_counts' is not defined

## Publisher Analysis

In [ ]:
# Count articles per publisher
publisher_counts = df['publisher'].value_counts()
print(publisher_counts.head(10))  # Top 10 publishers

# Visualize publisher contribution
publisher_counts.head(10).plot(kind="bar")
plt.title("Top 10 Publishers by Number of Articles")
plt.xlabel("Publisher")
plt.ylabel("Number of Articles")
plt.show()